# feder-kg master pipeline: publications → ontology-constrained knowledge graph

This notebook runs the whole document-processing pipeline with **Claude** as the extraction model.
It is a thin orchestrator: the logic lives in `src/` (`src/extract.py`, `src/ontology_loader.py`, `src/load_graph.py`
and helpers in `src/pipeline/`), so `python -m src.extract` runs the same thing without the notebook.

```
PDF ─► 1 ingest ─► 2 structure ─► T tables (Claude vision) ─► 3 pass 1: samples & process history
                                                              ─► 4 pass 2: measurements & values
                                                              ─► 5 pass 3: other entities & relations
    ─► 6 validate (code checks) ─► 7 resolve ─► 8 export (graph.json / CSV / Turtle / Neo4j) ─► 9 evaluate
```

| Stage | Module | Idea taken from the course notebooks |
|---|---|---|
| Ingest + structure | `src/pipeline/ingest.py`, `structure.py` | *Publication structure*: typography-relative headings, column-aware reading order, nothing silently dropped, section roles |
| Cleaning | `src/pipeline/clean.py` | *Murari*: explicit, documented cleanup (ligatures, hyphenation), raw text kept |
| Ontology gap terms, search | `src/pipeline/triage.py` | *Murari*: RAKE-style phrases, BM25 |
| Extraction | `src/extract.py`, `src/pipeline/schemas.py`, `llm.py` | *Spangher*: output contract first, evidence quotes, polarity, positive + negative examples, rejected candidates, content-addressed cache |
| Tables | `src/extract.py` | *Publication structure*: send only the hard regions (table pages) to a vision model |
| Validation | `src/pipeline/validate.py` | *Spangher*: quote anchoring without fuzzy repair; `structurally valid` ≠ `verified` |

What is new for this project: classes and relations are constrained by **your ontology** (`ontology/`, sliced by `ontology/slice_manifest.json`),
samples are tracked with their **process history**, values are **unit-normalized**, and there is an
**evaluation** step against your hand-annotated papers.

**Every stage saves `data/extractions/<paper_id>/<stage>.json` and is skipped on re-run if its inputs didn't change.
Every Claude response is cached by exact request, so re-running never pays twice.**

## 0. Setup (once)

1. From the repo root: `pip install -r requirements.txt`, then `docker compose up -d` if you want Neo4j.
2. Put your PDFs in `data/publications/` (not committed; their hashes go to `checksums.json`).
3. Put the ontology in `ontology/` (the file named in `slice_manifest.json` → `ontology_file`, by default
   `matsci-v10-semantic-closure.owl`). Until it is there, `ontology/starter_matsci_ontology.ttl` is used as a stand-in.
   Then check `ontology/slice_manifest.json` (which subtrees count as materials, processes, techniques, properties…).
4. In `.env` set `ANTHROPIC_API_KEY` (from console.anthropic.com). The key is never printed or saved.

In [7]:
import sys, json, os
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (PROJECT / "src" / "extract.py").exists(), "Open this notebook from the repo (notebooks/ or the repo root)."
sys.path.insert(0, str(PROJECT))

from src.pipeline.config import Config
from src.pipeline.runner import Pipeline, load_results, STAGES
from src.pipeline.triage import ontology_gap_terms, ParagraphSearch
from src.pipeline.evaluate import evaluate_paper, draft_gold_from_extraction
from src import extract as X

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 200)

cfg = Config(
    project_root=PROJECT,
    model="claude-sonnet-5-5",        # claude-haiku-5-5 = cheaper bulk runs; claude-opus-5-5 = hardest papers / evaluation
    max_output_tokens=16000,
    allow_live_calls=True,             # False = replay cached responses only (no API charges)
    min_confidence_for_auto_accept=0.85,
    extra={"use_vision_for_tables": True},
)
pl = Pipeline(cfg)
for n in cfg.notes:
    print("note:", n)
print("Ontology file:", cfg.ontology_file.name)
print("Claude:", pl.llm.status)
papers = pl.papers()
print(f"{len(papers)} PDFs in {cfg.papers_dir}")

Ontology file: matsci-v10-semantic-closure.owl
Claude: {'model': 'claude-sonnet-5-5', 'api_key_available': False, 'live_calls_enabled': True, 'mock': False}
0 PDFs in C:\Users\Jerry\Projects\synthgraph\data\publications


## 1. The ontology the extractor will use

The slice manifest (`ontology/slice_manifest.json`) turns subtrees of your ontology into the **only** class labels Claude may return (enums in the
structured-output schema). Warnings list missing roots and relations that have no object property in the ontology
yet. Both are to-dos for Protégé.

In [9]:
summary = pl.profile.summary()
display(pd.DataFrame([{"group": g, **v} for g, v in summary["groups"].items()]))
display(pd.DataFrame([{"relation": k, "maps_to": v} for k, v in summary["relations"].items()]))
for w in summary["warnings"]:
    print("WARNING:", w)

,group,allowed_classes,missing_roots
0,material,2,[]
1,sample,0,[sample]
2,process,480,[]
3,process_attribute,49,[]
4,technique,11,[]
5,property,269,[]
6,application,0,[application]
7,other,373,[]


,relation,maps_to
0,has_input,ontology: http://purl.obolibrary.org/obo/RO_0002233
1,has_output,ontology: http://purl.obolibrary.org/obo/RO_0002234
2,precedes,ontology: http://purl.obolibrary.org/obo/BFO_0000063
3,has_parameter,NOT IN ONTOLOGY (gap)
4,measured_on,ontology: http://purl.obolibrary.org/obo/OBI_0000293
5,has_property_value,ontology: http://purl.obolibrary.org/obo/RO_0000086
6,part_of,ontology: http://purl.obolibrary.org/obo/BFO_0000050
7,composed_of,ontology: http://purl.obolibrary.org/obo/BFO_0000051
8,intended_for,ontology: http://purl.obolibrary.org/obo/RO_0000085
9,influences,NOT IN ONTOLOGY (gap)


In [10]:
# The class tree as the pipeline sees it (like Protégé's hierarchy view)
print(pl.ontology.tree_text()[:4000])

chemical composition data item
  doping concentration
  measured composition
composition data item
Creep Strain
creep stress
cutting device
DEPRECATED aggregate state
DEPRECATED aggregate state value
DEPRECATED electrical property
DEPRECATED extensive quality
DEPRECATED intensive quality
DEPRECATED magnetic property
DEPRECATED mechanical property
DEPRECATED order scale
DEPRECATED order scale value
DEPRECATED stimulating process
DEPRECATED stimulation target role
DEPRECATED stimulus role
DEPRECATED test piece role
entity
  continuant
    charge carrier
    generically dependent continuant
      information content entity
        AnalysisInput
        AnalysisOutput
        AttributeAssertion
        avogadro constant
        boltzmann constant
        burgers circuit
        Chemical Class
        Chemical Family
        Chemical Formula
          Empirical Formula
          Molecular Formula
        Chemical Quality
          Chemical Family
          Chemical Formula
            Empir

## 2. Ingest and structure all papers (local, no API cost)

Checks to make here before spending anything on the API:
* **layout**: two-column pages should say `two_column`. If a two-column paper is read as single-column, the text interleaves.
* **roles found**: `methods` and `results` should be present. If headings are missed, look at `sections` for that paper.
* **warnings**: pages without text are scanned images and need OCR first.

In [12]:
run = pl.run_all(until="structure")
rows = []
for pid, r in run["results"].items():
    ing, st = r["ingest"], r["structure"]
    rows.append({
        "paper_id": pid, "pages": ing["source"]["page_count"],
        "layouts": ", ".join(sorted({p["layout"] for p in ing["pages"]})),
        "sections": len(st["sections"]), "paragraphs": len(st["paragraphs"]),
        "tables_parsed": sum(p["kind"] == "table" for p in st["paragraphs"]),
        "captions": sum(p["kind"] == "caption" for p in st["paragraphs"]),
        "roles": ", ".join(st["roles_found"]),
        "warnings": "; ".join(ing["warnings"] + st["warnings"]),
    })
structure_overview = pd.DataFrame(rows)
display(structure_overview)
print("Errors:", run["errors"] or "none")

[2023_QM012_Weak_anti_localization_effect_in_topological_Ni3In2S2_single_crystal.pdf]
[2023_QM027_Make_selenium_reactive_again_synthesis_of_metal_selenides.pdf]
[2023_QM034_Synthesis_of_ultra_pure_Co3Sn2S2_crystals.pdf]


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


[2023_QM045_Phosphorus_Solubility_and_Critical_Temperature_in_RbEuFe4As4xPx.pdf]
[2023_QM065_Direct_observation_of_topological_surface_states_in_layered_kagome_lattice_with_broken_time_reversal_symmetry.pdf]
[2024_QM003_A_single_crystal_study_of_Kagome_metals_U2Mn3Ge_and_U2Fe3Ge.pdf]
[2024_QM008_Molecular_beam_epitaxy_synthesis_and_electrical_transport_properties_of_Ni3In.pdf]
[2024_QM010_Synthesis_and_Stability_Phase_Diagram_of_Topological_Semimetal_Family_LnSbxTe2x.pdf]
[2024_QM017_Magnetic_Landscape_in_Antiferromagnetic_Kagome_Metal_TbTi3Bi4_and_Shurikagome_Metals.pdf]
[2024_QM050_Annealing_induced_long_range_charge_density_wave_order_in_magnetic_kagome_FeGe.pdf]
[2024_QM056_Enhancement_of_superconductivity_and_phase_diagram_of_Ta_doped_CsV3Sb5.pdf]
[2024_QM058_Synthesis_and_structure_of_a_non_van_der_Waals_two_dimensional_coordination_polymer_with_superconductivity.pdf]
[2024_QM062_Spectroscopic_origin_of_giant_anomalous_Hall_effect_in_an_interwoven_magnetic_kagome_metal.pdf]
[2024

,paper_id,pages,layouts,sections,paragraphs,tables_parsed,captions,roles,warnings
0,2023_QM012_Weak_anti_localization_effect_in_topological_Ni3In2S2_single_crystal,9,two_column,34,127,2,11,"abstract, acknowledgements, author_info, conclusion, front_matter, introduction, metho...",
1,2023_QM027_Make_selenium_reactive_again_synthesis_of_metal_selenides,32,single_or_uncertain,15,83,1,6,"abstract, acknowledgements, author_info, conclusion, front_matter, introduction, metho...",page 31: no extractable text (scanned page? needs OCR)
2,2023_QM034_Synthesis_of_ultra_pure_Co3Sn2S2_crystals,20,single_or_uncertain,17,87,0,0,"abstract, acknowledgements, author_info, front_matter, methods, other, references",
3,2023_QM045_Phosphorus_Solubility_and_Critical_Temperature_in_RbEuFe4As4xPx,23,"single_or_uncertain, two_column",25,75,3,8,"abstract, acknowledgements, author_info, conclusion, front_matter, introduction, metho...",
4,2023_QM065_Direct_observation_of_topological_surface_states_in_layered_kagome_la,9,"single_or_uncertain, two_column",8,53,0,4,"acknowledgements, author_info, front_matter, methods, results",
5,2024_QM003_A_single_crystal_study_of_Kagome_metals_U2Mn3Ge_and_U2Fe3Ge,7,"single_or_uncertain, two_column",18,50,0,7,"abstract, acknowledgements, author_info, conclusion, front_matter, introduction, metho...",
6,2024_QM008_Molecular_beam_epitaxy_synthesis_and_electrical_transport_properties_,6,two_column,2,71,0,4,front_matter,No methods/experimental section detected; process extraction will scan all body text.
7,2024_QM010_Synthesis_and_Stability_Phase_Diagram_of_Topological_Semimetal_Family,8,"single_or_uncertain, two_column",34,71,0,4,"acknowledgements, conclusion, front_matter, introduction, methods, other, references, ...",
8,2024_QM017_Magnetic_Landscape_in_Antiferromagnetic_Kagome_Metal_TbTi3Bi4_and_Shu,15,two_column,15,90,0,7,"acknowledgements, conclusion, front_matter, introduction, methods, other, results",
9,2024_QM050_Annealing_induced_long_range_charge_density_wave_order_in_magnetic_ka,10,"single_or_uncertain, two_column",19,83,0,5,"front_matter, introduction, methods, results, supplementary",


Errors: none


In [13]:
# Inspect one paper's sections and paragraphs
PAPER = structure_overview.paper_id.iloc[0] if len(structure_overview) else None
st = run["results"][PAPER]["structure"]
print("TITLE:", st["title"])
display(pd.DataFrame(st["sections"]))
display(pd.DataFrame(st["paragraphs"])[["id", "role", "kind", "page", "text"]].head(40))

TITLE: Weak anti-localization effect in topological Ni3In2S2 single crystal


,id,heading,role,level,page_start,page_end
0,SEC00,(front matter),front_matter,0,1,1
1,SEC01,J Mater Sci: Mater Electron (2023) 34:2302,front_matter,1,1,1
2,SEC02,Published online:,front_matter,1,1,1
3,SEC03,1 Introduction,introduction,1,1,1
4,SEC04,ABSTRACT,abstract,1,1,2
5,SEC05,2 Experimental details,methods,1,2,3
6,SEC06,3 Results and discussion,results,1,3,4
7,SEC07,(a),results,1,4,4
8,SEC08,(b),results,1,4,4
9,SEC09,70,results,1,4,4


,id,role,kind,page,text
0,P0001,front_matter,text,1,"Weak anti-localization effect in topological Ni3In2S2 single crystal Kapil Kumar^{1,2}..."
1,P0002,front_matter,text,1,"^{1} CSIR-National Physical Laboratory, Dr. K. S. Krishnan Marg, New Delhi 110012, Ind..."
2,P0003,front_matter,text,1,13 December 2023
3,P0004,front_matter,text,1,"© The Author(s), under exclusive licence to Springer Science+Business Media, LLC, part..."
4,P0005,abstract,text,1,"Ni3In2S2 is the most recent (Fang in Sci. China Mater. 66:2032, 2023 and Zhang in Npj ..."
5,P0006,abstract,text,1,"by the combination of different elements present in its structure. Nickel, as a transi..."
6,P0007,abstract,text,1,Address correspondence to E-mail: awana@nplindia.org
7,P0008,abstract,text,1,https://doi.org/10.1007/s10854-023-11756-1 Vol.:(0123456789) 2 302 Page 2 of 9
8,P0009,abstract,text,2,"condensed matter physics [2, 4]. These materials are of interest because they can exhi..."
9,P0010,abstract,text,2,In this context we report the growth and characterization of Ni3In2S2 single crystal t...


### 2.1 Ontology development aids (Murari-style, local)

Frequent technical phrases from the methods and results sections that match **no** class label or synonym in your
ontology. Use the list to decide which classes or synonyms to add in Protégé. It's a candidate list, not a verdict.

In [14]:
structures = {pid: r["structure"] for pid, r in run["results"].items()}
display(pd.DataFrame(ontology_gap_terms(structures, pl.ontology, top=40)))

,phrase,count,papers
0,consistent,135,38
1,samples,135,30
2,sample,128,33
3,inset,108,26
4,presence,107,30
5,data,104,29
6,cid:1,91,5
7,supplementary,77,11
8,figs,77,17
9,measurements,74,26


In [15]:
# BM25 search across all papers' paragraphs (useful to check coverage before/after extraction)
search = ParagraphSearch(structures)
display(pd.DataFrame(search.search("annealing temperature carrier lifetime passivation", k=8)))

,paper_id,paragraph_id,role,page,score,text
0,2024_QM050_Annealing_induced_long_range_charge_density_wave_order_in_magnetic_ka,P0030,results,5,9.724,We also tried annealing the as-grown samples under different temperatures from 523 to ...
1,2026_QM039_Electrochemical_Growth_of_Full_Volume_Meissner_Effect_Superconducting,P0042,introduction,3,9.448,reported. The post-growth annealing in oxygen is an effective approach that can largel...
2,2026_QM101_Metastable_MnBiTe_enabled_by_magnetic_field_assisted_synthesis,P0040,introduction,3,9.086,field annealing of MnBi2Te4 at 600 K has been shown to destabilize the A-type AFM ground
3,2026_QM039_Electrochemical_Growth_of_Full_Volume_Meissner_Effect_Superconducting,P0036,introduction,3,9.060,of techniques. While historically it was first synthesized using electrochemical growt...
4,2024_QM050_Annealing_induced_long_range_charge_density_wave_order_in_magnetic_ka,P0027,results,4,7.753,Annealing can be used as an effective way to reduce the concentration of defects in th...
5,2026_QM092_Topological_Surface_States_and_Anisotropic_Magnetotransport_in_SnSb6T,P0059,introduction,4,7.377,FIG. 4. (a) Temperature dependence of longitudinal resistance of SnSb6Te10. The red li...
6,2026_QM101_Metastable_MnBiTe_enabled_by_magnetic_field_assisted_synthesis,P0379,references,21,7.276,"[28] A. Rajbanshi, J. Xing, D. Gong, W. A. Shelton, and R. Jin, An Alternative Ground ..."
7,2023_QM027_Make_selenium_reactive_again_synthesis_of_metal_selenides,P0051,results,21,7.117,� 3𝑚𝑚 section HRTEM image of individual nanosheets (Figure 8b) viewed along c-axis ind...


## 3. Cost check, then a pilot on 1–2 papers

The paper text is sent once per pass (three passes), but it is marked for **prompt caching**, so passes 2 and 3
read it from the cache at a fraction of the input price. Table pages are sent as images. The estimate below is rough
(about 4 characters per token). Check current prices at docs.claude.com before large runs.

In [16]:
est = []
for pid, r in run["results"].items():
    docs = X.build_documents(r["structure"], r["structure"]["title"], cfg.skip_sections, cfg.max_chars_per_call)
    chars = sum(d["chars"] for d in docs)
    est.append({"paper_id": pid, "chunks": len(docs), "doc_chars": chars, "approx_doc_tokens": chars // 4,
                "table_pages_for_vision": len(X.table_pages(r["structure"])) if cfg.extra.get("use_vision_for_tables") else 0})
est = pd.DataFrame(est)
display(est)
print("System prompt (rules + ontology glossary): ~", len(pl.system) // 4, "tokens")
print("Total document tokens (one pass):", int(est.approx_doc_tokens.sum()))

,paper_id,chunks,doc_chars,approx_doc_tokens,table_pages_for_vision
0,2023_QM012_Weak_anti_localization_effect_in_topological_Ni3In2S2_single_crystal,1,25914,6478,4
1,2023_QM027_Make_selenium_reactive_again_synthesis_of_metal_selenides,1,52559,13139,1
2,2023_QM034_Synthesis_of_ultra_pure_Co3Sn2S2_crystals,1,37729,9432,0
3,2023_QM045_Phosphorus_Solubility_and_Critical_Temperature_in_RbEuFe4As4xPx,1,29138,7284,3
4,2023_QM065_Direct_observation_of_topological_surface_states_in_layered_kagome_la,1,26589,6647,0
5,2024_QM003_A_single_crystal_study_of_Kagome_metals_U2Mn3Ge_and_U2Fe3Ge,1,21288,5322,0
6,2024_QM008_Molecular_beam_epitaxy_synthesis_and_electrical_transport_properties_,1,33042,8260,0
7,2024_QM010_Synthesis_and_Stability_Phase_Diagram_of_Topological_Semimetal_Family,1,43187,10796,0
8,2024_QM017_Magnetic_Landscape_in_Antiferromagnetic_Kagome_Metal_TbTi3Bi4_and_Shu,1,77202,19300,0
9,2024_QM050_Annealing_induced_long_range_charge_density_wave_order_in_magnetic_ka,1,47727,11931,1


System prompt (rules + ontology glossary): ~ 44983 tokens
Total document tokens (one pass): 559673


In [ ]:
PILOT = papers[:1]          # start with one paper; read the results carefully before running all 50
pilot = pl.run_all(papers=PILOT, until="resolve")
print("Errors:", pilot["errors"] or "none")
display(pl.llm.usage_table())

### 3.1 Read the pilot output against the PDF

Look for: samples that should be split (different process histories), steps in the wrong order, values attached to
the wrong sample, wrong classes, and **things that are missing**. Fixes usually go into
`data/fewshot_examples.json` (add a positive and a *wrong* example), the profile, or the ontology. Re-running
then re-extracts only what changed.

In [ ]:
r = next(iter(pilot["results"].values()))   # the first pilot paper
p1, p2, p3 = r["pass1"], r["pass2"], r["pass3"]
print("Samples"); display(pd.DataFrame(p1["samples"])[["id", "name_in_paper", "ontology_class", "description", "material_ids", "confidence"]])
steps = pd.DataFrame([{**{k: s[k] for k in ("id", "ontology_class", "name_in_text", "step_order", "input_ids", "output_sample_ids")},
                       "parameters": "; ".join(f"{p['attribute_class']}={p['value_text']}" for p in s["parameters"])}
                      for s in p1["process_steps"]])
print("Process steps"); display(steps)
print("Unresolved notes:", p1["unresolved_notes"])

In [ ]:
cols = ["id", "technique_class", "property_class", "sample_ids", "value_text", "unit", "conditions", "data_origin", "polarity", "source_location", "confidence"]
print("Measurements"); display(pd.DataFrame(p2["measurements"])[cols])
print("Relations"); display(pd.DataFrame(p3["relations"])[["subject_id", "predicate", "object_id", "trend", "scope", "polarity", "support_type", "confidence"]] if p3["relations"] else "none")
print("Rejected candidates (kept on purpose)"); display(pd.DataFrame(p3["rejected_candidates"]))

## 4. Validation: what the code could and couldn't confirm

* `rejected`: no quote could be found in the paper, or an ID doesn't exist. Not exported.
* `needs_review`: anchored, but something deserves a human look (OTHER class, cited/computed value, hypothetical,
  number not found in the quoted paragraph, ontology gap, low confidence).
* `auto_accepted`: passed every check **and** confidence ≥ threshold. Still `verified_semantic_truth = False`.

In [ ]:
val = r["validate"]
display(pd.DataFrame(val["summary"]).pivot(index="kind", columns="status", values="count").fillna(0).astype(int))
problems = [{"kind": it["kind"], "id": it["id"], "status": it["status"], "issues": "; ".join(it["issues"] + it["notes"]),
             "quote": (it["evidence_anchors"][0]["quote"] if it["evidence_anchors"] else "")}
            for it in val["items"] if it["status"] != "auto_accepted"]
display(pd.DataFrame(problems))

## 5. Run all papers

Failures are collected per paper and don't stop the run. Re-running this cell resumes where it stopped: finished
stages are skipped and Claude calls replay from the cache.

In [ ]:
full = pl.run_all(until="resolve")
print("Papers done:", len(full["results"]), "| errors:", len(full["errors"]))
for k, v in full["errors"].items():
    print(" ", k, "->", v)
usage = pl.llm.usage_table()
if len(usage):
    print(usage.drop(columns=["label"]).groupby("cached").sum(numeric_only=True))

## 6. Export the knowledge graph

Writes to `data/graph/`:
* `nodes.jsonl`, `relationships.jsonl`: property graph in the same APOC JSON-lines format as `feder-graph/`. Every node and edge carries `paper_id`, `status`, `confidence`, `evidence`.
* `graph.ttl`: RDF typed with your ontology's class IRIs (can also be imported with Neo4j's neosemantics plugin).
* `review_queue.csv`: everything not auto-accepted, with the quote and page, plus empty `reviewer_decision` / `reviewer_comment` columns.
* `ontology_gaps.csv`: OTHER classes proposed by Claude, aggregated across papers. This is your to-do list for Protégé.
* `run_manifest.json`: config, software versions, ontology summary, and PDF hashes, for reproducibility.

Materials are merged across papers by (ontology class, normalized name). Samples, process steps and measurements stay
paper-specific.

In [ ]:
results = load_results(cfg.outputs_dir)   # everything on disk, including earlier runs
info = pl.export(results)
print(info)
display(pd.read_csv(cfg.graph_dir / "ontology_gaps.csv", encoding="utf-8-sig").head(30))
display(pd.read_csv(cfg.graph_dir / "review_queue.csv", encoding="utf-8-sig").head(20))

### 6.1 Load into Neo4j (optional)

Start Neo4j (`docker compose up -d`), check `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` in `.env`, then set
`LOAD_NEO4J = True`. This does the same as `python -m src.ontology_loader` (constraints + ontology class hierarchy)
followed by `python -m src.load_graph --replace`. All nodes get the label `KGNode` (unique `kg_key`) plus a type label
(`Sample`, `ProcessStep`, `Measurement`, …).

Example queries once loaded:

```cypher
// Process conditions behind lifetimes above 1 ms (experimental values only)
MATCH (m:Measurement)-[:MEASURED_ON]->(s:Sample)<-[:HAS_OUTPUT]-(p:ProcessStep)-[:HAS_PARAMETER]->(x:Parameter)
WHERE m.property = 'carrier lifetime' AND m.value_si > 1e-3 AND m.data_origin = 'experimental'
RETURN m.paper_id, s.name, p.name, x.attribute, x.value_text, m.value_text ORDER BY m.value_si DESC;

// Everything typed under a class, using the ontology hierarchy
MATCH (n)-[:INSTANCE_OF]->(:OntologyClass)-[:SUBCLASS_OF*0..]->(c:OntologyClass {name: 'deposition process'})
RETURN n.paper_id, n.name, n.ontology_class;

// Ontology gaps: instances Claude could not type
MATCH (n:KGNode) WHERE n.proposed_class IS NOT NULL RETURN n.proposed_class, count(*) AS n ORDER BY n DESC;
```

In [ ]:
LOAD_NEO4J = False
if LOAD_NEO4J:
    from src.ontology_loader import create_neo4j_schema
    from src.load_graph import driver_from_env, load_jsonl
    with driver_from_env() as driver:
        print(create_neo4j_schema(pl.ontology, driver))
        print(load_jsonl(driver, cfg.graph_dir / "nodes.jsonl", cfg.graph_dir / "relationships.jsonl", replace=True))

## 7. Evaluate against your hand-annotated papers

Without this step you can't tell whether a prompt or ontology change helped. Annotate 8–10 papers (see
`data/gold/README.md`), keep them out of the few-shot examples, and re-run this section after every change. Report
regressions as well as gains.

In [ ]:
# Write a draft gold file for a paper you are about to annotate (then correct it by hand and save as data/gold/<paper_id>.json)
DRAFT_FOR = None   # e.g. "Smith_2021_PECVD_SiNx"
if DRAFT_FOR:
    print(draft_gold_from_extraction(results[DRAFT_FOR]["resolve"], cfg.gold_dir / f"_draft_{DRAFT_FOR}.json"))

In [ ]:
scores = []
for gold_file in sorted(cfg.gold_dir.glob("*.json")):
    if gold_file.name.startswith("_"):
        continue
    pid = gold_file.stem
    if pid not in results or "resolve" not in results[pid]:
        print("No extraction for", pid); continue
    s = evaluate_paper(results[pid]["resolve"], json.loads(gold_file.read_text(encoding="utf-8")))["scores"]
    for k, v in s.items():
        if isinstance(v, dict):
            scores.append({"paper_id": pid, "item": k, **{m: v[m] for m in ("precision", "recall", "f1", "tp", "predicted", "gold")}})
        else:
            scores.append({"paper_id": pid, "item": k, "precision": v})
scores = pd.DataFrame(scores)
if len(scores):
    display(scores)
    display(scores.groupby("item")[["precision", "recall", "f1"]].mean())
else:
    print("No gold files yet. See data/gold/README.md.")

## 8. The iteration loop

1. **Read** the pilot output and the review queue against the PDFs.
2. **Fix the cause:**
   * wrong or missing classes → the ontology in Protégé, or `ontology/slice_manifest.json`
   * recurring judgment mistakes → a positive *and* a wrong example in `data/fewshot_examples.json`
   * layout/heading problems → `src/pipeline/ingest.py` / `structure.py` (check section 2 first)
3. **Re-run.** Only affected stages re-run; unchanged Claude calls replay from the cache.
4. **Evaluate** on the gold papers and keep the change only if it helps.

Known limits: numbers shown only in figures are not extracted (by design). Scanned PDFs need OCR. Model
confidence is not calibrated. Cross-paper merging covers materials only; sample/process equivalence across papers is
a research question for later (e.g. similar recipes).